# Project Contribution Summary

This notebook summarizes my work on adapting audio-language models for dementia-speech transcription.

Key contributions:

- Analyzed the processed train/test dataset used for model training and evaluation.
- Compared transcription baselines against fine-tuned audio-language models.
- Iterated on training strategies to improve both WER and disfluency-token recovery.
- Identified the training strategies that yielded the largest gains — synthetic disfluency-rich data, token weighting, and audio augmentation — leading to the best held-out WER and disfluency-token recovery in this experiment set.

# Processed Dataset Analysis

In [1]:
import subprocess
import sys
from pathlib import Path

# Install the project package from the repo (notebooks/../AudioLLM-D).
AUDIOLLM_D_DIR = (Path("..") / "AudioLLM-D").resolve()
if not (AUDIOLLM_D_DIR / "pyproject.toml").exists():
    raise FileNotFoundError(
        f"Expected AudioLLM-D at {AUDIOLLM_D_DIR}. "
        "Run this notebook with working directory set to notebooks/."
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(AUDIOLLM_D_DIR)],
    check=True,
)
print(f"Installed audiollm-d from {AUDIOLLM_D_DIR}")

Obtaining file:///scratch/sathvik/open-project-m2-lordofthestrings/AudioLLM-D
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for audiollm-d (pyproject.toml): started
  Building editable for audiollm-d (pyproject.toml): finished with status 'done'
  Created wheel for audiollm-d: filename=audiollm_d-0.1.0-0.editable-py3-none-any.whl size=3013 sha256=348f67ab9ce966f413079b6ae48dde7786d28bc67149ea2e0ee9957695e0661e
  Stored in directory: /tmp/pip-ephem-wheel-cache-oa47y5ew/wheels/23/b8/74/702917bdca33fde214a8c7eda481773f33a64b6e0

In [2]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd
import soundfile as sf

from audiollm_d.chat_parser import _CHAT_TOKEN_MAP

In [3]:
RESULTS_DIR = Path("sathvik")
SPLITS = ["train", "test"]

split_manifest_paths = {
    "train": RESULTS_DIR / "dataset_manifest_train.jsonl",
    "test": RESULTS_DIR / "dataset_manifest_test.jsonl",
}

status_df = pd.DataFrame(
    {
        "split": SPLITS,
        "manifest_path": [str(split_manifest_paths[s]) for s in SPLITS],
        "manifest_exists": [split_manifest_paths[s].exists() for s in SPLITS],
    }
)

display(status_df)

if not status_df["manifest_exists"].all():
    missing = status_df.loc[~status_df["manifest_exists"], "manifest_path"].tolist()
    raise FileNotFoundError(f"Missing manifests: {missing}")

,split,manifest_path,manifest_exists
0,train,sathvik/dataset_manifest_train.jsonl,True
1,test,sathvik/dataset_manifest_test.jsonl,True


In [4]:
def read_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def audio_duration_seconds(path: str | Path) -> float:
    info = sf.info(str(path))
    return float(info.frames) / float(info.samplerate)


def compile_token_pattern(token: str) -> re.Pattern[str]:
    # Count token as a standalone lexical unit, allowing punctuation around it.
    # This avoids under-counting forms like "uh," or "[short-pause]."
    return re.compile(rf"(?<![A-Za-z0-9_]){re.escape(token)}(?![A-Za-z0-9_])")


chat_source_tokens = list(_CHAT_TOKEN_MAP.keys())
mapped_tokens = sorted(set(_CHAT_TOKEN_MAP.values()))
mapped_token_patterns = {tok: compile_token_pattern(tok) for tok in mapped_tokens}
map_sub_pattern = re.compile(
    r"(?<![A-Za-z0-9_])(" 
    + "|".join(re.escape(token) for token in sorted(chat_source_tokens, key=len, reverse=True))
    + r")(?![A-Za-z0-9_])"
)


def apply_chat_map(text: str) -> str:
    return map_sub_pattern.sub(lambda m: _CHAT_TOKEN_MAP[m.group(1)], str(text))

In [5]:
split_records: dict[str, list[dict]] = {
    split: read_jsonl(path) for split, path in split_manifest_paths.items()
}

# Audio analysis for BOTH splits.
audio_summary_rows = []
for split, rows in split_records.items():
    durations = [audio_duration_seconds(r["audio_path"]) for r in rows]
    s = pd.Series(durations, dtype=float)
    audio_summary_rows.append(
        {
            "split": split,
            "num_clips": int(s.size),
            "total_seconds": float(s.sum()),
            "total_hours": float(s.sum() / 3600.0),
            "mean_seconds": float(s.mean()),
            "median_seconds": float(s.median()),
            "min_seconds": float(s.min()),
            "max_seconds": float(s.max()),
        }
    )

audio_summary_df = pd.DataFrame(audio_summary_rows).sort_values("split").reset_index(drop=True)

display(audio_summary_df)

,split,num_clips,total_seconds,total_hours,mean_seconds,median_seconds,min_seconds,max_seconds
0,test,168,3843.650029,1.067681,22.878869,26.35449,0.710998,29.96
1,train,658,15831.826906,4.397730,24.060527,26.57551,0.684989,30.00


In [6]:
# Processed text analysis for BOTH splits.
# Counts are computed only from the processed train/test manifests in merged_data_80_20.

mapped_rows = []
for split, rows in split_records.items():
    processed_texts = [apply_chat_map(str(r.get("text", ""))) for r in rows]
    split_mapped_counts = {
        token: sum(len(pattern.findall(text)) for text in processed_texts)
        for token, pattern in mapped_token_patterns.items()
    }

    for token in mapped_tokens:
        mapped_rows.append(
            {
                "split": split,
                "mapped_token": token,
                "count": int(split_mapped_counts[token]),
            }
        )

mapped_long_df = pd.DataFrame(mapped_rows)
mapped_dist_df = (
    mapped_long_df.pivot(index="mapped_token", columns="split", values="count")
    .fillna(0)
    .astype(int)
    .rename(columns={"train": "train_count", "test": "test_count"})
    .reset_index()
)

# Exclude lexical-normalization outputs from the processed disfluency-token table.
excluded_mapped_tokens = {"like", "you know"}
mapped_dist_df = mapped_dist_df[~mapped_dist_df["mapped_token"].isin(excluded_mapped_tokens)].copy()

mapped_dist_df["total_count"] = mapped_dist_df["train_count"] + mapped_dist_df["test_count"]
mapped_dist_df = mapped_dist_df.sort_values(["total_count", "mapped_token"], ascending=[False, True]).reset_index(drop=True)

print("Processed manifest disfluency-token distribution (train vs test)")
display(mapped_dist_df)

Processed manifest disfluency-token distribution (train vs test)


split,mapped_token,test_count,train_count,total_count
0,[uh],164,1127,1291
1,[unclear-speech],132,945,1077
2,[um],79,410,489
3,[short-pause],105,354,459
4,[long-pause],96,233,329
5,[medium-pause],53,143,196
6,[laughter],40,93,133
7,[non-verbal],24,87,111


## Disfluency token percentage in corpus

In [7]:
disfluency_tokens = [
    "[uh]",
    "[um]",
    "[short-pause]",
    "[medium-pause]",
    "[long-pause]",
    "[unclear-speech]",
    "[laughter]",
    "[non-verbal]",
]

disfluency_patterns = {tok: compile_token_pattern(tok) for tok in disfluency_tokens}

split_text_stats = []
per_token_rows = []
combined_texts: list[str] = []

for split, rows in split_records.items():
    mapped_texts = [apply_chat_map(str(r.get("text", ""))) for r in rows]
    combined_texts.extend(mapped_texts)

    total_tokens = sum(len(text.split()) for text in mapped_texts)
    token_counts = {
        tok: sum(len(disfluency_patterns[tok].findall(text)) for text in mapped_texts)
        for tok in disfluency_tokens
    }
    total_disfluency_tokens = sum(token_counts.values())

    split_text_stats.append(
        {
            "split": split,
            "num_samples": len(rows),
            "total_tokens": total_tokens,
            "total_disfluency_tokens": total_disfluency_tokens,
            "disfluency_token_percentage": (100.0 * total_disfluency_tokens / total_tokens) if total_tokens else 0.0,
        }
    )

    for tok in disfluency_tokens:
        count = token_counts[tok]
        per_token_rows.append(
            {
                "split": split,
                "token": tok,
                "count": count,
                "pct_of_all_tokens": (100.0 * count / total_tokens) if total_tokens else 0.0,
                "pct_within_disfluency_tokens": (
                    100.0 * count / total_disfluency_tokens
                )
                if total_disfluency_tokens
                else 0.0,
            }
        )

# Add combined (train + test)
combined_total_tokens = sum(len(text.split()) for text in combined_texts)
combined_counts = {
    tok: sum(len(disfluency_patterns[tok].findall(text)) for text in combined_texts)
    for tok in disfluency_tokens
}
combined_total_disfluency = sum(combined_counts.values())

split_text_stats.append(
    {
        "split": "all",
        "num_samples": len(combined_texts),
        "total_tokens": combined_total_tokens,
        "total_disfluency_tokens": combined_total_disfluency,
        "disfluency_token_percentage": (
            100.0 * combined_total_disfluency / combined_total_tokens
        )
        if combined_total_tokens
        else 0.0,
    }
)

for tok in disfluency_tokens:
    count = combined_counts[tok]
    per_token_rows.append(
        {
            "split": "all",
            "token": tok,
            "count": count,
            "pct_of_all_tokens": (100.0 * count / combined_total_tokens) if combined_total_tokens else 0.0,
            "pct_within_disfluency_tokens": (100.0 * count / combined_total_disfluency)
            if combined_total_disfluency
            else 0.0,
        }
    )

corpus_disfluency_df = pd.DataFrame(split_text_stats)
per_token_disfluency_df = pd.DataFrame(per_token_rows)

print("Overall disfluency-token percentage by split")
display(corpus_disfluency_df.style.format({"disfluency_token_percentage": "{:.2f}%"}))

print("Per-token distribution (percentage of corpus and of disfluency tokens)")
display(
    per_token_disfluency_df.sort_values(["split", "token"]).reset_index(drop=True).style.format(
        {
            "pct_of_all_tokens": "{:.3f}%",
            "pct_within_disfluency_tokens": "{:.2f}%",
        }
    )
)

Overall disfluency-token percentage by split


,split,num_samples,total_tokens,total_disfluency_tokens,disfluency_token_percentage
0,train,658,32461,3392,10.45%
1,test,168,6913,693,10.02%
2,all,826,39374,4085,10.37%


Per-token distribution (percentage of corpus and of disfluency tokens)


,split,token,count,pct_of_all_tokens,pct_within_disfluency_tokens
0,all,[laughter],133,0.338%,3.26%
1,all,[long-pause],329,0.836%,8.05%
2,all,[medium-pause],196,0.498%,4.80%
3,all,[non-verbal],111,0.282%,2.72%
4,all,[short-pause],459,1.166%,11.24%
5,all,[uh],1291,3.279%,31.60%
6,all,[um],489,1.242%,11.97%
7,all,[unclear-speech],1077,2.735%,26.36%
8,test,[laughter],40,0.579%,5.77%
9,test,[long-pause],96,1.389%,13.85%


## Key Insights

- The dataset is small: 658 training clips and 168 test clips, with about 4.4 hours of training audio and 1.1 hours of test audio.
- Most clips are short, averaging around 23-24 seconds and capped at about 30 seconds, so a small number of missed markers can noticeably affect token-level F1.
- Disfluency markers make up about 10% of corpus tokens, but are concentrated in a few frequent tokens such as `[uh]`, `[unclear-speech]`, `[um]`, and pause tokens.
- Rare tokens such as `[laughter]` and `[non-verbal]` have limited support, motivating token weighting, data augmentation, and other strategies to learn them well.

# Model Peformance Analysis

## Model Comparison

This section follows the fine-tuning progression in the order it was explored. The WER table includes external ASR baselines and the base model before training, while the F1 tables focus on LoRA models because only they emit the bracketed disfluency-token labels.

Progression of experiments:

- `LLM-only LoRA`: Fine-tuned only the language-model side. This tested whether the model could learn the disfluency-token format mainly through generation behavior. It improved over the base model, but the gap to the best run showed that audio-side adaptation was also important.
- `LLM + MM Projector LoRA`: Also fine-tuned the multimodal projector that maps audio representations into the language model. This tested whether improving the audio-to-language interface was enough, but it still did not fully close the gap.
- `Plain LoRA`: Fine-tuned the standard LoRA setup on the original processed data. This became the main starting point because it showed that LoRA fine-tuning could recover bracketed disfluency tokens much better than generic ASR baselines.
- `+ Token-Weighted Augmented LoRA`: Added inverse weighting for rare disfluency tokens and audio augmentation. The augmentation pipeline used white noise, simulated babble, synthetic reverb, speed perturbation, and codec-compression effects, directly targeting class imbalance and limited acoustic variation in the small training set.
- `+ MFA + Classification Head LoRA` (side experiment): Used MFA alignments to train an auxiliary classifier for disfluency locations. The classifier outputs were added back to the encoder outputs to bias, or align, the representation toward disfluency-relevant audio regions. This was not carried forward because it performed worse than the token-weighted augmented model, possibly due to noisy MFA labels.
- `+ CrisperWhisper Encoder LoRA`: Swapped in the [CrisperWhisper](https://huggingface.co/nyrahealth/CrisperWhisper) speech encoder to test whether stronger acoustic representations improved disfluency recovery. This helped some token-level behavior, but was not the final best run.
- `+ Synthetic Data LoRA`: Added synthetic disfluency-rich examples on top of the original data. The synthetic clips were generated with neural TTS using the [Chatterbox TTS](https://github.com/resemble-ai/chatterbox) package by creating short spoken contexts and inserting explicit disfluency events: true silence for pause tokens, degraded speech for `[unclear-speech]`, generated or event-bank laughter/non-verbal sounds, and filler tokens such as `[uh]` and `[um]`. Voice, rate, and pitch were varied to avoid a single synthetic speaking style. This was the best final model I trained, with the lowest held-out WER and strongest aggregate disfluency-token F1 in this comparison.

In [8]:
RESULTS_DIR = Path("sathvik")

model_metric_paths = {
    "LLM-only LoRA": RESULTS_DIR / "eval_metrics_llm_only_lora.json",
    "LLM + MM Projector LoRA": RESULTS_DIR / "eval_metrics_llm_projector_lora.json",
    "Plain LoRA": RESULTS_DIR / "eval_metrics_plain_lora.json",
    "+ Token-Weighted Augmented LoRA": RESULTS_DIR / "eval_metrics_token_weighted_augmented_lora.json",
    "+ MFA + Classification Head LoRA": RESULTS_DIR / "eval_metrics_mfa_classification_head_lora.json",
    "+ CrisperWhisper Encoder LoRA": RESULTS_DIR / "eval_metrics_crisperwhisper_encoder_lora.json",
    "+ Synthetic Data LoRA": RESULTS_DIR / "eval_metrics_synthetic_data_lora_best.json",
}

baseline_metric_paths = {
    "Whisper Large baseline": {
        "train": RESULTS_DIR / "eval_metrics_whisper_large_baseline_train.json",
        "test": RESULTS_DIR / "eval_metrics_whisper_large_baseline_test.json",
    },
    "CrisperWhisper baseline": {
        "train": RESULTS_DIR / "eval_metrics_crisperwhisper_baseline_train.json",
        "test": RESULTS_DIR / "eval_metrics_crisperwhisper_baseline_test.json",
    },
}

model_metrics = {}
for model_name, metrics_path in model_metric_paths.items():
    with metrics_path.open("r", encoding="utf-8") as f:
        model_metrics[model_name] = json.load(f)

baseline_metrics = {}
for model_name, split_paths in baseline_metric_paths.items():
    baseline_metrics[model_name] = {}
    for split, metrics_path in split_paths.items():
        with metrics_path.open("r", encoding="utf-8") as f:
            baseline_metrics[model_name][split] = json.load(f)


def final_split_metrics(metrics: dict, split: str) -> dict:
    """Return final train/test metrics, handling runs that call train `original_train`."""
    split_options = {
        "train": ["train", "original_train"],
        "test": ["test", "validation"],
    }[split]

    after_train = metrics["after_train"]
    for split_key in split_options:
        if split_key in after_train:
            return after_train[split_key]
    raise KeyError(f"Could not find {split!r} metrics in after_train keys: {list(after_train)}")


def disfluency_metrics(metrics: dict, split: str) -> dict:
    return final_split_metrics(metrics, split)["disfluency_token_metrics"]


def per_token_metric_map(metrics: dict, split: str, order_aware: bool = False) -> dict:
    token_metrics = disfluency_metrics(metrics, split)
    key = "per_token_order_aware_stats" if order_aware else "per_token_stats"
    return token_metrics.get(key, {})


def split_label(split: str) -> str:
    return split.title()


# 1) WER-only summary for every model, including ASR baselines.
wer_display_order = [
    "Whisper Large baseline",
    "CrisperWhisper baseline",
    "Base model (before training)",
    "LLM-only LoRA",
    "LLM + MM Projector LoRA",
    "Plain LoRA",
    "+ Token-Weighted Augmented LoRA",
    "+ MFA + Classification Head LoRA (Side Experiment)",
    "+ CrisperWhisper Encoder LoRA",
    "+ Synthetic Data LoRA",
]
wer_order_map = {model_name: i for i, model_name in enumerate(wer_display_order)}

wer_rows = []
plain_lora_metrics = model_metrics["Plain LoRA"]
base_row = {"Model": "Base model (before training)", "Type": "Untrained LoRA/base AudioLLM"}
for split in ["train", "test"]:
    block = plain_lora_metrics["before_train"][split]
    prefix = split_label(split)
    base_row[f"{prefix} Samples"] = block["num_samples"]
    base_row[f"{prefix} WER"] = block["wer"]
wer_rows.append(base_row)

for model_name, metrics in model_metrics.items():
    row = {"Model": model_name, "Type": "LoRA fine-tune"}
    for split in ["train", "test"]:
        block = final_split_metrics(metrics, split)
        prefix = split_label(split)
        row[f"{prefix} Samples"] = block["num_samples"]
        row[f"{prefix} WER"] = block["wer"]
    wer_rows.append(row)

for model_name, split_metrics in baseline_metrics.items():
    row = {"Model": model_name, "Type": "ASR baseline"}
    for split in ["train", "test"]:
        block = split_metrics[split]
        prefix = split_label(split)
        row[f"{prefix} Samples"] = block["num_samples"]
        row[f"{prefix} WER"] = block["wer"]
    wer_rows.append(row)

wer_summary_df = pd.DataFrame(wer_rows)
wer_summary_df["Sort Order"] = wer_summary_df["Model"].map(wer_order_map)
wer_summary_df = wer_summary_df.sort_values("Sort Order").drop(columns="Sort Order").reset_index(drop=True)

wer_format = {
    "Train WER": "{:.4f}",
    "Test WER": "{:.4f}",
}

print("WER summary in model-progression order")
display(wer_summary_df.style.format(wer_format))


# 2) Aggregate WER + F1 table for LoRA runs only.
aggregate_rows = []
for model_name, metrics in model_metrics.items():
    for split in ["train", "test"]:
        block = final_split_metrics(metrics, split)
        d = block["disfluency_token_metrics"]
        aggregate_rows.append(
            {
                "Model": model_name,
                "Split": split_label(split),
                "Samples": block["num_samples"],
                "WER": block["wer"],
                "F1": d.get("disfluency_token_f1"),
                "F1 (order-aware)": d.get("disfluency_token_order_f1"),
            }
        )

aggregate_model_order = {model_name: i for i, model_name in enumerate(model_metrics)}
aggregate_model_comparison_df = pd.DataFrame(aggregate_rows)
aggregate_model_comparison_df["Model Order"] = aggregate_model_comparison_df["Model"].map(aggregate_model_order)
aggregate_model_comparison_df = (
    aggregate_model_comparison_df
    .sort_values(["Split", "Model Order"])
    .drop(columns="Model Order")
    .reset_index(drop=True)
)

aggregate_format = {
    "WER": "{:.4f}",
    "F1": "{:.4f}",
    "F1 (order-aware)": "{:.4f}",
}

lora_train_aggregate_df = aggregate_model_comparison_df[
    aggregate_model_comparison_df["Split"] == "Train"
].drop(columns="Split").reset_index(drop=True)
lora_test_aggregate_df = aggregate_model_comparison_df[
    aggregate_model_comparison_df["Split"] == "Test"
].drop(columns="Split").reset_index(drop=True)

print("\nLoRA aggregate metrics (Train)")
display(lora_train_aggregate_df.style.format(aggregate_format))

print("\nLoRA aggregate metrics (Test)")
display(lora_test_aggregate_df.style.format(aggregate_format))


# 3) Token-level F1 tables for LoRA runs only; ASR baselines do not emit the same bracketed tokens.
def build_token_f1_table(split: str, order_aware: bool = False) -> pd.DataFrame:
    all_tokens = sorted(
        set().union(
            *[
                set(per_token_metric_map(metrics, split, order_aware=order_aware).keys())
                for metrics in model_metrics.values()
            ]
        )
    )

    rows = []
    for token in all_tokens:
        row = {"Token": token}
        for model_name, metrics in model_metrics.items():
            token_stats = per_token_metric_map(metrics, split, order_aware=order_aware).get(token)
            row[model_name] = None if token_stats is None else token_stats.get("f1", 0.0)
        rows.append(row)

    return pd.DataFrame(rows).sort_values("Token").reset_index(drop=True)


token_f1_format = {model_name: "{:.4f}" for model_name in model_metrics}
for split in ["test", "train"]:
    print(f"\nToken-level F1 by LoRA model ({split_label(split)}, no order)")
    display(build_token_f1_table(split, order_aware=False).style.format(token_f1_format, na_rep=""))

    print(f"\nToken-level F1 by LoRA model ({split_label(split)}, order-aware)")
    display(build_token_f1_table(split, order_aware=True).style.format(token_f1_format, na_rep=""))

WER summary in model-progression order


,Model,Type,Train Samples,Train WER,Test Samples,Test WER
0,Whisper Large baseline,ASR baseline,658,0.4601,168,0.5141
1,CrisperWhisper baseline,ASR baseline,658,0.4222,168,0.4665
2,Base model (before training),Untrained LoRA/base AudioLLM,658,0.4932,168,0.5688
3,LLM-only LoRA,LoRA fine-tune,658,0.0838,168,0.4649
4,LLM + MM Projector LoRA,LoRA fine-tune,658,0.0381,168,0.4902
5,Plain LoRA,LoRA fine-tune,658,0.0048,168,0.3692
6,+ Token-Weighted Augmented LoRA,LoRA fine-tune,658,0.0212,168,0.3803
7,+ CrisperWhisper Encoder LoRA,LoRA fine-tune,658,0.0208,168,0.3828
8,+ Synthetic Data LoRA,LoRA fine-tune,658,0.0410,168,0.3566
9,+ MFA + Classification Head LoRA,LoRA fine-tune,658,0.0437,168,0.3653



LoRA aggregate metrics (Train)


,Model,Samples,WER,F1,F1 (order-aware)
0,LLM-only LoRA,658,0.0838,0.8748,0.8546
1,LLM + MM Projector LoRA,658,0.0381,0.9384,0.9323
2,Plain LoRA,658,0.0048,0.9895,0.9876
3,+ Token-Weighted Augmented LoRA,658,0.0212,0.9406,0.9351
4,+ MFA + Classification Head LoRA,658,0.0437,0.9396,0.9340
5,+ CrisperWhisper Encoder LoRA,658,0.0208,0.9663,0.9627
6,+ Synthetic Data LoRA,658,0.0410,0.9251,0.9167



LoRA aggregate metrics (Test)


...

,Token,LLM-only LoRA,LLM + MM Projector LoRA,Plain LoRA,+ Token-Weighted Augmented LoRA,+ MFA + Classification Head LoRA,+ CrisperWhisper Encoder LoRA,+ Synthetic Data LoRA
0,[laughter],0.0000,0.0690,0.0000,0.2759,0.0870,0.2308,0.0741
1,[long-pause],0.1745,0.1486,0.3797,0.3247,0.2297,0.3919,0.3478
2,[medium-pause],0.1707,0.1266,0.1622,0.1370,0.1579,0.1644,0.2353
3,[non-verbal],0.2083,0.0606,0.2778,0.2162,0.1053,0.0976,0.1000
4,[short-pause],0.2022,0.1808,0.2086,0.1724,0.1609,0.1576,0.2118
5,[uh],0.5430,0.5724,0.6421,0.6207,0.5775,0.6330,0.6090
6,[um],0.7059,0.7051,0.7305,0.7389,0.7226,0.7403,0.7643
7,[unclear-speech],0.0932,0.1122,0.1201,0.1717,0.1436,0.1910,0.1809



Token-level F1 by LoRA model (Train, no order)


,Token,LLM-only LoRA,LLM + MM Projector LoRA,Plain LoRA,+ Token-Weighted Augmented LoRA,+ MFA + Classification Head LoRA,+ CrisperWhisper Encoder LoRA,+ Synthetic Data LoRA
0,[laughter],0.8837,0.9444,0.9946,0.9836,0.9551,0.9892,0.9392
1,[long-pause],0.8818,0.9321,0.9826,0.9713,0.9497,0.9782,0.9448
2,[medium-pause],0.9077,0.9500,0.9930,0.9485,0.9574,0.9412,0.9084
3,[non-verbal],0.7667,0.9586,0.9942,0.9461,0.9091,0.9595,0.9341
4,[short-pause],0.8624,0.9384,0.9857,0.9469,0.9457,0.9443,0.9050
5,[uh],0.9164,0.9599,0.9942,0.9410,0.9377,0.9768,0.9629
6,[um],0.8053,0.9319,0.9890,0.9716,0.9645,0.9707,0.9494
7,[unclear-speech],0.8667,0.9127,0.9858,0.9121,0.9242,0.9586,0.8719



Token-level F1 by LoRA model (Train, order-aware)


,Token,LLM-only LoRA,LLM + MM Projector LoRA,Plain LoRA,+ Token-Weighted Augmented LoRA,+ MFA + Classification Head LoRA,+ CrisperWhisper Encoder LoRA,+ Synthetic Data LoRA
0,[laughter],0.8358,0.9091,1.0000,0.9714,0.9565,0.9863,0.9143
1,[long-pause],0.8591,0.9276,0.9826,0.9713,0.9453,0.9782,0.9360
2,[medium-pause],0.9077,0.9517,0.9925,0.9453,0.9556,0.9375,0.9105
3,[non-verbal],0.7215,0.9595,0.9934,0.9379,0.8951,0.9600,0.9178
4,[short-pause],0.8418,0.9351,0.9857,0.9383,0.9276,0.9384,0.8961
5,[uh],0.9005,0.9562,0.9942,0.9401,0.9341,0.9745,0.9583
6,[um],0.8018,0.9282,0.9890,0.9691,0.9546,0.9706,0.9383
7,[unclear-speech],0.8290,0.8969,0.9782,0.8978,0.9212,0.9512,0.8534


## Key Insights

- Generic ASR baselines are useful for word-level comparison, but they are not designed to preserve the bracketed disfluency vocabulary. The LoRA results are therefore more informative for judging whether the model learned the project-specific output format.
- The base AudioLLM did not naturally generate the disfluency labels, so fine-tuning was necessary before the model could recover pauses, fillers, unclear speech, and other markers in a reliable way.
- Plain LoRA established the core result: the model can learn to emit the bracketed disfluency tokens from the processed dementia-speech data.
- Token weighting and augmentation were most useful as targeted fixes for the dataset imbalance. They encouraged the model to pay more attention to rare markers, even when the overall WER did not improve.
- The MFA/classification-head branch was a useful side experiment, but it added complexity without clear benefit. This suggests that the alignment labels or auxiliary classifier signal were not reliable enough to improve the final transcription behavior.
- Changing the encoder to CrisperWhisper showed that stronger acoustic features can help some disfluency-token behavior, but encoder quality alone was not the full solution.
- The best overall result came from adding synthetic disfluency-rich examples. Qualitatively, this makes sense: the model needed more exposure to the exact marker patterns it was expected to generate, not just more parameter updates.
- The final model is the strongest one because it combines improved transcription with the best overall disfluency-token recovery on the held-out set.

## Predictions of the Best Model

The cells below show curated train and test examples from the saved prediction files. Each test example is chosen to illustrate a specific pattern in the token-level analysis: strong filler recovery, near-perfect held-out transcription, or characteristic failure modes such as dropped pauses, missing laughter/non-verbal events, pause-category confusion, and transcript drift.

In [9]:
import json
from pathlib import Path

from IPython.display import Markdown, display

RESULTS_DIR = Path("sathvik")
BEST_MODEL_PREDICTION_PATHS = {
    "train": RESULTS_DIR / "predictions_synthetic_data_lora_best_train.jsonl",
    "test": RESULTS_DIR / "predictions_synthetic_data_lora_best_test.jsonl",
}


def read_prediction_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


best_model_predictions = {
    split: read_prediction_jsonl(path)
    for split, path in BEST_MODEL_PREDICTION_PATHS.items()
}


def normalize_prediction_selection(selection, *, n_rows: int) -> list[int]:
    """Accept a single index, list/tuple/set of indices, range, or slice."""
    if isinstance(selection, int):
        indices = [selection]
    elif isinstance(selection, range):
        indices = list(selection)
    elif isinstance(selection, slice):
        indices = list(range(n_rows))[selection]
    elif isinstance(selection, (list, tuple, set)):
        indices = list(selection)
    else:
        raise TypeError("Use an int, list/tuple/set of ints, range(...), or slice(...).")

    clean_indices = []
    for idx in indices:
        if not isinstance(idx, int):
            raise TypeError(f"Prediction index must be an int, got {type(idx).__name__}: {idx!r}")
        if idx < 0 or idx >= n_rows:
            raise IndexError(f"Index {idx} out of range for {n_rows} predictions")
        clean_indices.append(idx)
    return clean_indices


def show_best_model_predictions(split: str, selection) -> None:
    split = split.lower()
    if split not in best_model_predictions:
        raise ValueError(f"Unknown split {split!r}; use one of {list(best_model_predictions)}")

    rows = best_model_predictions[split]
    indices = normalize_prediction_selection(selection, n_rows=len(rows))
    sections = []

    for idx in indices:
        row = rows[idx]
        reference = row.get("reference", "")
        prediction = row.get("prediction", "")
        sections.append(
            f"""
### {split.title()} prediction #{idx}
**Ground truth**

{reference}

**Prediction**

{prediction}
""".strip()
        )

    display(Markdown("\n\n---\n\n".join(sections)))


for split, rows in best_model_predictions.items():
    print(f"{split.title()}: {len(rows)} predictions loaded from {BEST_MODEL_PREDICTION_PATHS[split]}")


def score_prediction_example(reference: str, prediction: str) -> dict:
    """Return per-example WER and order-aware disfluency-token metrics."""
    from audiollm_d.metrics import compute_order_aware_disfluency_token_metrics, compute_wer
    from audiollm_d.special_tokens import DISFLUENCY_TOKENS

    order_metrics = compute_order_aware_disfluency_token_metrics(
        [reference], [prediction], DISFLUENCY_TOKENS
    )
    return {
        "wer": compute_wer([reference], [prediction]),
        "order_f1": order_metrics["disfluency_token_order_f1"],
        "order_tp": order_metrics["disfluency_token_order_tp"],
        "order_ref": order_metrics["disfluency_token_order_ref_total"],
        "order_pred": order_metrics["disfluency_token_order_pred_total"],
    }


def show_annotated_predictions(split: str, examples: list[dict]) -> None:
    """Display predictions with a short justification for each selected index."""
    split = split.lower()
    rows = best_model_predictions[split]
    sections = []

    for example in examples:
        idx = example["idx"]
        row = rows[idx]
        reference = row.get("reference", "")
        prediction = row.get("prediction", "")
        metrics = score_prediction_example(reference, prediction)
        sections.append(
            f"""
### {split.title()} prediction #{idx} — {example['label']}

{example['note']}

**Metrics:** WER={metrics['wer']:.3f}, order-aware disfluency F1={metrics['order_f1']:.3f} ({metrics['order_tp']}/{metrics['order_ref']} tokens matched in order)

**Ground truth**

{reference}

**Prediction**

{prediction}
""".strip()
        )

    display(Markdown("\n\n---\n\n".join(sections)))

Train: 658 predictions loaded from sathvik/predictions_synthetic_data_lora_best_train.jsonl
Test: 168 predictions loaded from sathvik/predictions_synthetic_data_lora_best_test.jsonl


In [10]:
# Strong training-set examples: the model can recover many marker types when the clip is seen during training.
train_examples = [
    {
        "idx": 0,
        "label": "Near-perfect training fit",
        "note": "Exact match on `[um]`, `[long-pause]`, `[non-verbal]`, and `[medium-pause]`. This supports the claim that frequent markers are learned well on training data.",
    },
    {
        "idx": 217,
        "label": "Dense marker recovery on train",
        "note": "Recovers a long chain of `[uh]`, `[um]`, `[short-pause]`, `[long-pause]`, and `[unclear-speech]` with order-aware F1=1.0, showing the model can handle heavily disfluent training clips.",
    },
]

show_annotated_predictions("train", train_examples)

### Train prediction #0 — Near-perfect training fit

Exact match on `[um]`, `[long-pause]`, `[non-verbal]`, and `[medium-pause]`. This supports the claim that frequent markers are learned well on training data.

**Metrics:** WER=0.000, order-aware disfluency F1=1.000 (9/9 tokens matched in order)

**Ground truth**

okay [um] [long-pause] this is the [long-pause] Cinderella , yes. and these are the stepmothers. oh , and they went to the ball for the [long-pause] and she hadta stay at home. and so they're going and having [um] [um] [long-pause] oh , [non-verbal] [medium-pause] yeah.

**Prediction**

okay [um] [long-pause] this is the [long-pause] Cinderella , yes. and these are the stepmothers. oh , and they went to the ball for the [long-pause] and she hadta stay at home. and so they're going and having [um] [um] [long-pause] oh , [non-verbal] [medium-pause] yeah.

---

### Train prediction #217 — Dense marker recovery on train

Recovers a long chain of `[uh]`, `[um]`, `[short-pause]`, `[long-pause]`, and `[unclear-speech]` with order-aware F1=1.0, showing the model can handle heavily disfluent training clips.

**Metrics:** WER=0.000, order-aware disfluency F1=1.000 (21/21 tokens matched in order)

**Ground truth**

[uh] she told them that that [uh] she needed to [uh] [short-pause] [uh] allow [short-pause] [uh] [unclear-speech] Cinderella [long-pause] and [uh] so she's [um] [short-pause] the [uh] the the the the [short-pause] [um] [long-pause] [uh] I I I don't know. [um] but as the the folks from [laughter] [long-pause] [uh] I don't know.

**Prediction**

[uh] she told them that that [uh] she needed to [uh] [short-pause] [uh] allow [short-pause] [uh] [unclear-speech] Cinderella [long-pause] and [uh] so she's [um] [short-pause] the [uh] the the the the [short-pause] [um] [long-pause] [uh] I I I don't know. [um] but as the the folks from [laughter] [long-pause] [uh] I don't know.

In [11]:
# Curated held-out examples chosen to justify the token-level takeaways below.
test_examples = [
    {
        "idx": 5,
        "label": "Strong held-out filler recovery",
        "note": "Perfect match on both `[uh]` and `[um]` with WER=0.00. This is one of the best test cases and supports the claim that fillers generalize better than pauses or non-verbal sounds.",
    },
    {
        "idx": 53,
        "label": "Strong held-out filler recovery (dense clip)",
        "note": "All six filler tokens match in order (three `[um]`, three `[uh]`) with order-aware F1=1.0. Minor punctuation/word edits only.",
    },
    {
        "idx": 120,
        "label": "Good held-out `[uh]` recovery",
        "note": "Both `[uh]` markers are preserved exactly; only a small lexical change (`drying dry` → `trying to dry`).",
    },
    {
        "idx": 0,
        "label": "Cleaner transcript / dropped non-verbal",
        "note": "The sentence stays fluent, but `[non-verbal]` and `[medium-pause]` are dropped while `[uh]` and `[um]` remain. This illustrates the model simplifying annotation when the words are still understandable.",
    },
    {
        "idx": 1,
        "label": "Pause-category confusion",
        "note": "The model keeps pauses but changes category: `[long-pause]` becomes `[medium-pause]`, and both `[short-pause]` markers disappear. This supports the pause-duration error pattern.",
    },
    {
        "idx": 17,
        "label": "Dropped laughter and non-verbal events",
        "note": "Reference has six `[laughter]`, one `[non-verbal]`, and one `[medium-pause]`; prediction keeps only one `[laughter]` and swaps `[medium-pause]` → `[short-pause]`. Many disfluency events are removed from an otherwise readable transcript.",
    },
    {
        "idx": 18,
        "label": "Short-pause loss and `[uh]` → `[um]` swap",
        "note": "Reference contains `[short-pause]`, `[medium-pause]`, `[long-pause]`, `[laughter]`, and `[non-verbal]`; prediction drops most of them and changes the opening `[uh]` to `[um]`.",
    },
    {
        "idx": 90,
        "label": "`[unclear-speech]` error modes",
        "note": "Opening `[short-pause]` is lost, four `[unclear-speech]` markers become zero, and the model inserts `[long-pause]` / `[medium-pause]` instead. This shows both under-prediction and fallback pause insertion.",
    },
    {
        "idx": 25,
        "label": "Transcript drift in a highly disfluent clip",
        "note": "Once lexical content drifts (`mice`, `win`, `hat`), pause alignment collapses: `[long-pause]` becomes `[short-pause]`, several pauses disappear, and order-aware F1=0.0 despite some surviving markers.",
    },
    {
        "idx": 152,
        "label": "Severe transcript drift",
        "note": "The prediction abandons the reference structure entirely (WER≈1.46), producing unrelated sentences while only loosely keeping topic words like `boy on a stool`. Disfluency alignment fails when the word sequence diverges.",
    },
]

show_annotated_predictions("test", test_examples)

### Test prediction #5 — Strong held-out filler recovery

Perfect match on both `[uh]` and `[um]` with WER=0.00. This is one of the best test cases and supports the claim that fillers generalize better than pauses or non-verbal sounds.

**Metrics:** WER=0.000, order-aware disfluency F1=1.000 (2/2 tokens matched in order)

**Ground truth**

and [uh] she recognizes Cinderella recognizes it. and [um] it's a perfect fit. and they live happily ever after. because the prince recognizes her.

**Prediction**

and [uh] she recognizes Cinderella recognizes it. and [um] it's a perfect fit. and they live happily ever after. because the prince recognizes her.

---

### Test prediction #53 — Strong held-out filler recovery (dense clip)

All six filler tokens match in order (three `[um]`, three `[uh]`) with order-aware F1=1.0. Minor punctuation/word edits only.

**Metrics:** WER=0.047, order-aware disfluency F1=1.000 (6/6 tokens matched in order)

**Ground truth**

and [um] the sink is overflowing , creating a large pool of water on the floor. [uh] the mother is [um] doesn't seem to realize that this is happening. [um] she is standing in the water. and she is [uh] drying a plate. in the meantime , right behind her , the [uh] boy is on a three legged stool which is tipping over.

**Prediction**

and [um] the sink is overflowing creating a large pool of water on the floor. [uh] the mother is [um] doesn't seem to realize that this is happening. [um] she is standing in the water. and she is [uh] drying a plate. in the meantime right behind her the [uh] boy is on a three legged stool which is tipping over.

---

### Test prediction #120 — Good held-out `[uh]` recovery

Both `[uh]` markers are preserved exactly; only a small lexical change (`drying dry` → `trying to dry`).

**Metrics:** WER=0.035, order-aware disfluency F1=1.000 (2/2 tokens matched in order)

**Ground truth**

okay the kid on the bench who's got his hand in the cookie jar and he's falling off and his sister wants one. [uh] his mother is standing in a puddle of water because she didn't turn off the faucet and she's drying dry a dish. she oughta dry her feet instead. [uh] the window is open.

**Prediction**

okay the kid on the bench who's got his hand in the cookie jar and he's falling off and his sister wants one. [uh] his mother is standing in a puddle of water because she didn't turn off the faucet and she's trying to dry a dish. she oughta dry her feet instead. [uh] the window is open.

---

### Test prediction #0 — Cleaner transcript / dropped non-verbal

The sentence stays fluent, but `[non-verbal]` and `[medium-pause]` are dropped while `[uh]` and `[um]` remain. This illustrates the model simplifying annotation when the words are still understandable.

**Metrics:** WER=0.143, order-aware disfluency F1=0.667 (2/4 tokens matched in order)

**Ground truth**

[uh] this is Cinderella. those are her two stepsisters and the wicked stepmother. [non-verbal] and [um] [medium-pause] she:'s washing the floor.

**Prediction**

[uh] this is Cinderella. those are her two stepsisters and the wicked stepmother. and [um] she's washing the floor.

---

### Test prediction #1 — Pause-category confusion

The model keeps pauses but changes category: `[long-pause]` becomes `[medium-pause]`, and both `[short-pause]` markers disappear. This supports the pause-duration error pattern.

**Metrics:** WER=0.273, order-aware disfluency F1=0.615 (4/8 tokens matched in order)

**Ground truth**

[uh] because [uh] [short-pause] the wicked stepsisters and stepmother [um] [short-pause] tell her what to do. and [um] [long-pause] let's see [long-pause]

**Prediction**

[uh] because [uh] the wicked stepisters and stepmother [um] tell her what to do. and [um] [medium-pause] let's see.

---

### Test prediction #17 — Dropped laughter and non-verbal events

Reference has six `[laughter]`, one `[non-verbal]`, and one `[medium-pause]`; prediction keeps only one `[laughter]` and swaps `[medium-pause]` → `[short-pause]`. Many disfluency events are removed from an otherwise readable transcript.

**Metrics:** WER=0.567, order-aware disfluency F1=0.118 (1/13 tokens matched in order)

**Ground truth**

[laughter] go backwards , okay. [laughter] oh okay. [non-verbal] [short-pause] this is really good. oh , I can't do it. [laughter] I really I? I'm [uh] [laughter] [unclear-speech]. twenty [medium-pause] [unclear-speech] [uh] twenty nine. thirty. I'm doing it am I doing it in the wrong [laughter] [medium-pause]? out comes backwards [laughter]. mhm. we can [uh] [unclear-speech] stop there. can you find it another [laughter]? mhm. she's Cinderella.

**Prediction**

[unclear-speech] go backwards okay? oh okay. this is really good. no I can't do it. I'm sure. I am not. [um] I'm doing it. am I doing it on the wrong [short-pause] thing? you can [uh] stop there. can you find it another? huh? she was a Cinderella.

---

### Test prediction #18 — Short-pause loss and `[uh]` → `[um]` swap

Reference contains `[short-pause]`, `[medium-pause]`, `[long-pause]`, `[laughter]`, and `[non-verbal]`; prediction drops most of them and changes the opening `[uh]` to `[um]`.

**Metrics:** WER=0.579, order-aware disfluency F1=0.167 (2/17 tokens matched in order)

**Ground truth**

and that's her [uh] [unclear-speech] [unclear-speech] mother. and these are their sisters. mhm. [um] [medium-pause] [um] Cinderella is still washing dish or a plate. and [um] [short-pause] and the mothers and brothers I mean , sisters [laughter] and then [um] [uh] [non-verbal] [uh] gave it to somebody [laughter] [long-pause] anyhow [uh] then she looked so [unclear-speech] [laughter].

**Prediction**

and that's her [um] [unclear-speech] mother. and these are their sisters. [non-verbal] [uh] [unclear-speech] Cinderella is still washing the floor. [unclear-speech] sisters and her brothers I think and sisters. anyhow [uh] then she looks so fat [laughter].

---

### Test prediction #90 — `[unclear-speech]` error modes

Opening `[short-pause]` is lost, four `[unclear-speech]` markers become zero, and the model inserts `[long-pause]` / `[medium-pause]` instead. This shows both under-prediction and fallback pause insertion.

**Metrics:** WER=1.080, order-aware disfluency F1=0.000 (0/3 tokens matched in order)

**Ground truth**

[short-pause] she's preparing a meal. I'm going to hafta quit reading this because this this is shining on this [unclear-speech] [unclear-speech] getting my [unclear-speech]. [unclear-speech].

**Prediction**

and she's carrying a little [long-pause] I don't know what else to put with this because it's it's turning on this on this on this on this [long-pause] drying my hand. [medium-pause] I don't know.

---

### Test prediction #25 — Transcript drift in a highly disfluent clip

Once lexical content drifts (`mice`, `win`, `hat`), pause alignment collapses: `[long-pause]` becomes `[short-pause]`, several pauses disappear, and order-aware F1=0.0 despite some surviving markers.

**Metrics:** WER=0.689, order-aware disfluency F1=0.000 (0/10 tokens matched in order)

**Ground truth**

people [long-pause] for a pumpkin. and it's wonderful. and lots of things. and there's three little [unclear-speech]. and bring [long-pause] her mother [long-pause] Cinderella and and her [non-verbal] [long-pause] Cinderella [unclear-speech] [long-pause] and [medium-pause] a mother with a [short-pause] when [long-pause] and that's her beautiful.

**Prediction**

people [short-pause] for a pumpkin. and it's with a new [unclear-speech]. and lots of things. and there's three little mice. and the mother [short-pause] with a [medium-pause] win. and hat is that her beautiful [non-verbal] [long-pause] okay , I think that's it.

---

### Test prediction #152 — Severe transcript drift

The prediction abandons the reference structure entirely (WER≈1.46), producing unrelated sentences while only loosely keeping topic words like `boy on a stool`. Disfluency alignment fails when the word sequence diverges.

**Metrics:** WER=1.464, order-aware disfluency F1=0.000 (0/1 tokens matched in order)

**Ground truth**

little girl with her finger to her lips. the boy on the stool. stool tipping over. getting cookies out_of the cookie jar. [uh] mother washing dishes. water running.

**Prediction**

see that. what issues with cleaning the kitchen? can't do it by themselves. you just repeat the sentences after me. boy on a stool. mom's mad at them because they broke the window. getting cookies on the teacher's desk. the mother washes the dishes for us. water running I don't know.

## Best Model Token Error Analysis

In [12]:
import json
from pathlib import Path

import pandas as pd

BEST_MODEL_METRICS_PATH = Path("sathvik/eval_metrics_synthetic_data_lora_best.json")

with BEST_MODEL_METRICS_PATH.open("r", encoding="utf-8") as f:
    best_model_metrics = json.load(f)


def saved_order_aware_token_table(metrics: dict, split: str) -> pd.DataFrame:
    token_stats = metrics["after_train"][split]["disfluency_token_metrics"]["per_token_order_aware_stats"]
    rows = []
    for token, stats in token_stats.items():
        correct = int(stats["tp"])
        reference_count = int(stats["ref_total"])
        prediction_count = int(stats["pred_total"])
        rows.append(
            {
                "Token": token,
                "Reference Count": reference_count,
                "Prediction Count": prediction_count,
                "Correct": correct,
                "Missed": reference_count - correct,
                "Extra/Wrong": prediction_count - correct,
                "Precision": stats["precision"],
                "Recall": stats["recall"],
                "F1": stats["f1"],
            }
        )

    token_order = {token: i for i, token in enumerate(token_stats)}
    df = pd.DataFrame(rows).sort_values("Token", key=lambda s: s.map(token_order)).reset_index(drop=True)
    df[["Precision", "Recall", "F1"]] = df[["Precision", "Recall", "F1"]].round(3)
    return df


train_token_errors_df = saved_order_aware_token_table(best_model_metrics, "original_train")
test_token_errors_df = saved_order_aware_token_table(best_model_metrics, "test")

print("Train token error analysis from saved metrics")
display(train_token_errors_df)

print("Test token error analysis from saved metrics")
display(test_token_errors_df)

Train token error analysis from saved metrics


,Token,Reference Count,Prediction Count,Correct,Missed,Extra/Wrong,Precision,Recall,F1
0,[short-pause],354,320,302,52,18,0.944,0.853,0.896
1,[medium-pause],135,122,117,18,5,0.959,0.867,0.911
2,[long-pause],233,220,212,21,8,0.964,0.910,0.936
3,[unclear-speech],821,796,690,131,106,0.867,0.840,0.853
4,[laughter],36,34,32,4,2,0.941,0.889,0.914
5,[non-verbal],76,70,67,9,3,0.957,0.882,0.918
6,[uh],1126,1080,1057,69,23,0.979,0.939,0.958
7,[um],409,401,380,29,21,0.948,0.929,0.938


Test token error analysis from saved metrics


,Token,Reference Count,Prediction Count,Correct,Missed,Extra/Wrong,Precision,Recall,F1
0,[short-pause],105,65,18,87,47,0.277,0.171,0.212
1,[medium-pause],51,34,10,41,24,0.294,0.196,0.235
2,[long-pause],96,65,28,68,37,0.431,0.292,0.348
3,[unclear-speech],109,90,18,91,72,0.200,0.165,0.181
4,[laughter],20,7,1,19,6,0.143,0.050,0.074
5,[non-verbal],24,16,2,22,14,0.125,0.083,0.100
6,[uh],162,127,88,74,39,0.693,0.543,0.609
7,[um],79,78,60,19,18,0.769,0.759,0.764


### Token-Level Takeaways

- The model fits the training set well: frequent tokens such as `[uh]`, `[um]`, and pause markers are recovered with high precision and recall. **Train #0** is an exact match on `[um]`, multiple `[long-pause]` markers, `[non-verbal]`, and `[medium-pause]`; **train #217** preserves a long chain of fillers, pauses, and `[unclear-speech]` with order-aware F1=1.0.
- Test performance is much weaker, showing that the remaining errors are mostly generalization errors rather than failures to learn the output format. Only **7/168** test clips reach order-aware F1=1.0, while many others still produce readable text but lose markers (e.g. **test #17**, order-aware F1=0.12).
- `[um]` is the most reliable held-out token. It has the best balance of precision and recall, suggesting the model learned this acoustic/textual pattern more consistently than the other markers. **Test #5** and **#53** are representative strong cases where every `[um]` and `[uh]` is preserved in order.
- `[uh]` is also comparatively strong, but it still has more misses than `[um]`, which suggests filler recovery is uneven across token types. In **test #18**, the opening `[uh]` becomes `[um]` even though surrounding content is partially preserved.
- Pause tokens are usually under-predicted on the test set. The model misses many `[short-pause]`, `[medium-pause]`, and `[long-pause]` markers, suggesting it can detect some silence events but does not reliably preserve every pause marker. **Test #0** drops `[medium-pause]`; **test #18** loses `[short-pause]`, `[medium-pause]`, and `[long-pause]`; **test #1** replaces `[long-pause]` with `[medium-pause]`.
- `[unclear-speech]` remains difficult despite being common. It has many misses and extra/wrong predictions, which suggests the model sometimes treats acoustically difficult regions as ordinary words and sometimes uses `[unclear-speech]` as a fallback when lexical recognition is uncertain. **Test #90** removes all four `[unclear-speech]` markers and substitutes pauses instead.
- `[laughter]` and `[non-verbal]` are the weakest held-out tokens. They are rare and have low recall, so the model usually misses them rather than confidently generating them. **Test #17** drops five of six `[laughter]` events and the sole `[non-verbal]` marker; **test #0** drops `[non-verbal]` entirely.

### Qualitative Error Patterns

- The model often makes the transcript cleaner than the reference. It drops pauses, fillers, and non-verbal events when the surrounding sentence is still understandable. **Test #0** keeps the Cinderella description fluent but removes `[non-verbal]` and `[medium-pause]`; **test #17** removes most laughter while leaving the narrative intact.
- Short pauses are especially easy to lose. In many examples, `[short-pause]` appears between fluent words in the reference, but the prediction connects the words directly. **Test #18** removes `[short-pause]` between `Cinderella` and `and the mothers`; **test #90** drops the opening `[short-pause]`.
- When the model does predict pauses, it sometimes gets the duration wrong. `[long-pause]` may become `[medium-pause]`, or `[medium-pause]` may become `[short-pause]`. **Test #1** changes `[long-pause]` → `[medium-pause]`; **test #17** changes `[medium-pause]` → `[short-pause]`; **test #25** repeatedly downgrades `[long-pause]` to `[short-pause]`.
- `[unclear-speech]` has two opposite error modes. Sometimes the model replaces unclear regions with a plausible word, such as turning an unclear span into a guessed lexical item. Other times it inserts `[unclear-speech]` near a difficult word, using the token as a fallback when it is uncertain about the transcription. **Test #90** shows the replacement mode (four `[unclear-speech]` markers disappear and are replaced by guessed words/pauses).
- Laughter and non-verbal sounds are usually omitted unless they are very salient. The model often keeps the surrounding words and simply skips `[laughter]` or `[non-verbal]`, which explains their low recall. **Test #17** is the clearest example: six laughter events shrink to one.
- Filler errors are more subtle than pause or non-verbal errors. `[uh]` and `[um]` are sometimes dropped at the beginning of a phrase, and they are occasionally swapped with each other. **Test #2** drops the opening `[uh]` markers but keeps later `[um]`; **test #18** swaps `[uh]` → `[um]`.
- The largest qualitative failures happen when the transcript drifts, especially in long or highly disfluent clips. Once the word sequence diverges from the reference, the disfluency markers also become misaligned, producing many token errors in a single example. **Test #25** (WER≈0.69, order-aware F1=0.0) and **test #152** (WER≈1.46) show this cascading failure mode most clearly.

# Conclusion

Fine-tuning clearly improved dementia-speech transcription in this project. The untrained base AudioLLM did not produce the bracketed disfluency vocabulary at all (0% disfluency-token F1 on the held-out set), while the best fine-tuned model reduced test WER from **0.57 → 0.36** and reached **0.57 disfluency-token F1** (0.40 order-aware). That is a meaningful gain over generic ASR baselines such as Whisper Large (test WER 0.51) and CrisperWhisper (0.47), and it shows that LoRA adaptation can teach the model both lexical content and the project-specific marker format.

At the same time, the problem remains difficult and far from solved. The dataset is small, only **658 training clips (~4.4 h)** and **168 test clips (~1.1 h)** and disfluency markers make up only about **10%** of tokens while being spread across many rare categories. With so little data, even strong train-set performance (best-model train WER **0.04**, disfluency-token F1 **0.93**) does not transfer cleanly: held-out WER stays at **0.36** and order-aware token F1 drops to **0.40**. The large train–test gap suggests the main remaining errors are generalization failures, not a failure to learn the output format.

The experiments also show that progress required more than scaling parameters. Plain LoRA established that the model could learn the marker vocabulary, but further gains came from targeted interventions for imbalance and exposure — token weighting, audio augmentation, and finally a synthetic-data curriculum that gave the model more examples of the exact patterns it was asked to generate. Even so, error analysis on the best model shows uneven held-out behavior: `[um]` and `[uh]` are recovered reasonably well, while pause tokens, `[unclear-speech]`, `[laughter]`, and `[non-verbal]` remain weak, often dropped or simplified when the surrounding sentence is still intelligible.

Overall, fine-tuning is necessary and effective for this task, but limited data, label sparsity, and the fine-grained nature of disfluency annotation make reliable held-out recovery hard. Future work likely needs more labeled dementia speech, better coverage of rare events which can improve generalization.